In [2]:
# Daily Challenge: Fine-Tuning GPT-2 for SMS Spam Classification (Legacy transformers API)
  # In this daily challenge, you’ll fine-tune a pre-trained GPT-2 model to classify SMS messages as spam or ham (not spam). We’ll work through loading the dataset, inspecting its schema, tokenizing examples, adapting to
  # an older transformers version, and running training and evaluation with the classic do_train/do_eval flags.

  # Contexte général :  Adapter un modèle GPT-2 pour qu’il ne génère pas de texte, mais qu’il classe des SMS comme "spam" ou "ham" (non spam). GPT-2 n'est pas fait pour ça à l’origine, donc on va lui ajouter une tête
  # de classification et entraîner ce nouveau modèle avec un dataset réel.

In [7]:
!pip install -U datasets

In [9]:
#  1. Setup : Install required packages datasets, evaluate and transformers[sentencepiece].     # installation des packages requis

%pip install --quiet datasets evaluate transformers[sentencepiece]


In [1]:
# 2. Load & Inspect Dataset :

from datasets import load_dataset
import pandas as pd

# Charger le jeu de données SMS Spam depuis le hub Hugging Face
dataset = load_dataset("ucirvine/sms_spam")

# Découper en jeu d'entraînement et validation
train_ds = dataset['train'].select(range(4000))
val_ds   = dataset['train'].select(range(4000, 5000))

# Afficher les colonnes/features du jeu d'entraînement
print(train_ds.features)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/359k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5574 [00:00<?, ? examples/s]

{'sms': Value('string'), 'label': ClassLabel(names=['ham', 'spam'])}


In [3]:
[row['sms'] for row in dataset['train'] if row['label'] == 1]

["Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's\n",
 "FreeMsg Hey there darling it's been 3 week's now and no word back! I'd like some fun you up for it still? Tb ok! XxX std chgs to send, £1.50 to rcv\n",
 'WINNER!! As a valued network customer you have been selected to receivea £900 prize reward! To claim call 09061701461. Claim code KL341. Valid 12 hours only.\n',
 'Had your mobile 11 months or more? U R entitled to Update to the latest colour mobiles with camera for Free! Call The Mobile Update Co FREE on 08002986030\n',
 'SIX chances to win CASH! From 100 to 20,000 pounds txt> CSH11 and send to 87575. Cost 150p/day, 6days, 16+ TsandCs apply Reply HL 4 info\n',
 'URGENT! You have won a 1 week FREE membership in our £100,000 Prize Jackpot! Txt the word: CLAIM to No: 81010 T&C www.dbuk.net LCCLTD POBOX 4403LDNW1A7RW18\n',
 'XXXMobileMovieClub: To use your credit, click the WA

In [4]:
#3. Tokenization

from transformers import GPT2Tokenizer

model_name = "gpt2"  # nom du modèle

tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 n’a pas de token [PAD]

# Fonction de tokenisation
def tokenize_fn(examples):
    return tokenizer(
        examples["sms"],
        padding="max_length",     # forcer la même longueur
        truncation=True,          # couper les messages trop longs
        max_length=64             # suffisant pour des SMS
    )

# Application de la fonction à chaque dataset
train_tok = train_ds.map(tokenize_fn, batched=True)
val_tok   = val_ds.map(tokenize_fn, batched=True)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [5]:
# 4. Model Initialization

import torch
from transformers import GPT2ForSequenceClassification

model = GPT2ForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,  # ✅ spam ou ham
    pad_token_id=tokenizer.eos_token_id  # nécessaire pour éviter erreur sur GPT-2
)

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
# 5. Metrics Definition

import evaluate
import numpy as np

# Charger les fonctions de métrique
accuracy  = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall    = evaluate.load("recall")
f1        = evaluate.load("f1")

# Fonction qui retourne les métriques pour chaque batch
def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy":  accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "precision": precision.compute(predictions=preds, references=labels)["precision"],
        "recall":    recall.compute(predictions=preds, references=labels)["recall"],
        "f1":        f1.compute(predictions=preds, references=labels)["f1"]
    }

  # In an imbalanced dataset like SMS spam (often more “ham” than “spam”), why is it important to track precision and recall alongside accuracy?
        # Si le modèle est très bon pour reconnaître le ham (majoritaire), il peut avoir une haute accuracy même s’il ignore les spams.
  # How would you interpret a model that achieves high accuracy but low recall on the spam class?
        # Le recall permet de savoir s’il attrape bien tous les spams. La précision indique si ce qu’il signale comme spam est réellement du spam.

In [13]:
# 6. TrainingArguments Configuration

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./gpt2-spam-model",      # ✅ dossier de sortie
    do_train=True,
    do_eval=True,
    eval_steps=500,                      # ✅ évaluer toutes les 500 étapes
    save_steps=500,                      # ✅ sauvegarder toutes les 500 étapes
    logging_dir="./logs",
    logging_steps=500,

    per_device_train_batch_size=8,       # ✅ (peut être ajusté selon RAM dispo)
    per_device_eval_batch_size=8,
    num_train_epochs=3,                  # ✅ ajustable
    learning_rate=5e-5,                  # ✅ valeur classique
    weight_decay=0.01,                   # ✅ régularisation

    report_to="none",                      # pas de logging externe
    save_total_limit=1,                  # garder 1 seul checkpoint
)

  # What effect does weight_decay have during fine-tuning? When might you choose a higher or lower value?
      # Il pénalise les poids trop grands, pour éviter le surapprentissage (overfitting). Plus sa valeur est haute, plus la régularisation est forte (peut ralentir/aplatir l’apprentissage).
      # Si le modèle surapprend trop vite : augmente le weight_decay. Si le modèle n’apprend pas assez vite : réduis weight_decay.

In [14]:
# 7. Train & Evaluate

# Train
from transformers import Trainer
# you need to have your wandb api key ready to paste in the command line
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    compute_metrics=compute_metrics,
)
trainer.train()             # entraînement

# Évaluation finale
metrics = trainer.evaluate()
print(metrics)

Step,Training Loss
500,0.096400
1000,0.031800
1500,0.006600


{'eval_loss': 0.05238202586770058, 'eval_accuracy': 0.994, 'eval_precision': 1.0, 'eval_recall': 0.9568345323741008, 'eval_f1': 0.9779411764705882, 'eval_runtime': 3.9383, 'eval_samples_per_second': 253.914, 'eval_steps_per_second': 31.739, 'epoch': 3.0}


# Interpret your results.

# 1. eval_loss = 0.0524 (erreur moyenne du modèle pendant l'évaluation). Valeur faible : le modèle fait très peu d'erreurs sur les prédictions.
# 2. eval_accuracy = 0.994 (99,4%) - Le modèle a correctement classé 99,4 % des SMS de l’ensemble de validation (ham ou spam). C’est très bien; mais ne suffit pas à évaluer un modèle sur un jeu de données déséquilibré (où "ham" est très majoritaire).
# 3. eval_precision = 1.0 (100%) - proportion de prédictions positives (spam) qui sont correctes. Tous les SMS classés comme spam l’étaient réellement → aucun faux positif.
# 4. eval_recall = 0.957 (95,7%) - Le recall mesure la proportion de spams correctement détectés. 95,7 % des vrais spams représente un très bon score; toutefois le modèle a laissé passer quelques spams (faux négatifs). Pour l'améliorer, on pourrait l'entraîner un peu plus longtemps (avec plus d’epochs), ajuster les poids de classe, ou augmenter la taille du dataset.
# 5. eval_f1 = 0.978 (moyenne harmonique entre précision et rappel). Très bon équilibre entre ces deux dimensions.
# 6. eval_runtime = 3.94 sec - Temps total pour évaluer le modèle sur l'ensemble de validation. C'est très rapide, car petit dataset (1000 SMS) et modèle léger (GPT-2 base).
# 7. eval_samples_per_second = 254 / eval_steps_per_second = 31.7. Donne des indications sur la vitesse de traitement pendant l'évaluation.
# 8. epoch = 3.0. Le modèle a été entraîné sur 3 passages complets du dataset d’entraînement.

# Pour conclure, nous avons entraîné un excellent classifieur de SMS spam :


Fine-Tuning GPT-2 for SMS Spam
Last Updated: May 10th, 2025

Daily Challenge: Fine-Tuning GPT-2 for SMS Spam Classification (Legacy transformers API)


In this daily challenge, you’ll fine-tune a pre-trained GPT-2 model to classify SMS messages as spam or ham (not spam). We’ll work through loading the dataset, inspecting its schema, tokenizing examples, adapting to an older transformers version, and running training and evaluation with the classic do_train/do_eval flags.



👩‍🏫 👩🏿‍🏫 What You’ll learn
How to load and explore a custom text-classification dataset
Inspecting and aligning column names for tokenization
Tokenizing text for GPT-2 (with its peculiar padding setup)
Initializing GPT2ForSequenceClassification
Defining and computing multiple evaluation metrics
Configuring TrainingArguments for transformers < 4.4 (using do_train, eval_steps, etc.)
Running fine-tuning with Trainer and interpreting results
Common pitfalls when using legacy APIs


🛠️ What you will create
By the end of this challenge, you will have built:

A tokenized SMS dataset compatible with GPT-2’s requirements, including custom padding and truncation.
A fine-tuned GPT2ForSequenceClassification model that can accurately label incoming SMS messages as spam or ham.
A complete training pipeline using the legacy do_train/do_eval flags in TrainingArguments, with periodic checkpointing, logging, and evaluation.
A set of evaluation metrics (accuracy, precision, recall, F1) computed at each validation step and summarized after training.
A reusable Jupyter notebook that ties everything together—from dataset loading and inspection, through model initialization and tokenization, to training, evaluation, and results interpretation.


💼 Prerequisites
Python 3.7+
Installed packages: datasets, evaluate, transformers>=4.0.0,<4.4.0
Basic familiarity with Hugging Face’s datasets and transformers libraries
GitHub or Colab access for executing the notebook
A Hugging Face API and a WeightAndBiases API, for instructions on how to get it, click here.


Task
We will guide you through making a fine-tuning a GPT-2 model to classify SMS messages as spam or ham using an older version of transformers (<4.4). Follow the steps below and complete the “TODO” in the code.

1. Setup : Install required packages datasets, evaluate and transformers[sentencepiece].

%pip install --quiet datasets evaluate transformers[sentencepiece]


2. Load & Inspect Dataset :

from datasets import TODO #import load_dataset
TODO # import pandas

# Load the UCI SMS Spam dataset (sms_spam) from Hugging Face hub
raw = TODO

# We'll use 4,000 for train, 1,000 for validation
train_ds = TODO
val_ds   = TODO

TODO  # print the features of the train dataset. It should show 'sms' and 'label'


3. Tokenization :

from transformers import TODO # import GPT2Tokenizer


model_name = TODO #load the tokenize, we will use GPT2
tokenizer  = TODO
# GPT-2 has no pad token by default—set it to eos
tokenizer.pad_token = tokenizer.eos_token

def tokenize_fn(examples):
    # returns input_ids, attention_mask; keep max_length small for SMS
    return tokenizer(
        examples["sms"],
        padding="max_length",
        truncation=True,
        max_length=64
    )

train_tok = TODO #apply the tokenization by loading the subset using .map function
val_tok   = TODO #apply the tokenization by loading the subset using .map function



4. Model Initialization

import torch
TODO  #import GPT2ForSequenceClassification

model = GPT2ForSequenceClassification.from_pretrained( # Load GPT-2 with sequence classification head
    model_name,
    num_labels=TODO,           # spam vs. ham
    pad_token_id=tokenizer.eos_token_id
)


5. Metrics Definition

import evaluate
import numpy as np

accuracy  = evaluate.load("accuracy")
precision = # apply the function used for accurracy but for precision
recall    = # apply the function used for accurracy but for recall
f1        = # apply the function used for accurracy but for F1

def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy":  accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "precision": TODO, # apply the function used for accurracy but for precision
        "recall":    TODO, # apply the function used for accurracy but for recall
        "f1":        TODO # apply the function used for accurracy but for F1
    }


In an imbalanced dataset like SMS spam (often more “ham” than “spam”), why is it important to track precision and recall alongside accuracy?
How would you interpret a model that achieves high accuracy but low recall on the spam class?


6. TrainingArguments Configuration

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=TODO
    do_train=True,                 # turn on training
    do_eval=True,                  # turn on evaluation
    eval_steps=TODO,                # run .evaluate() every 500 steps
    save_steps=TODO,                # save a checkpoint every 500 steps
    logging_dir="./logs",
    logging_steps=TODO,             # log metrics every 500 steps

    per_device_train_batch_size=TODO,
    per_device_eval_batch_size=TODO,
    num_train_epochs=TODO,
    learning_rate=TODO,
    weight_decay=TODO,

    report_to=None,                # disable integrations
    save_total_limit=1,            # only keep last checkpoint
)


What effect does weight_decay have during fine-tuning? When might you choose a higher or lower value?


7. Train & Evaluate

# Train
from transformers import Trainer
# you need to have your wandb api key ready to paste in the command line
trainer = Trainer(
    model=TODO,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    compute_metrics=compute_metrics,
)
trainer.train()

#Evaluate
metrics = TODO
print(metrics)
# Expect something like: {"eval_loss": ..., "eval_accuracy": 0.98, ...}



Interpret your results.